# Suite BLW — A store given a store below

A store may be given the store below it, from which it loads what it is asked for, when it is asked ([CAP-283: a store given a store below](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-283), [CAP-284: objects loaded on demand](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-284)): `Builders.Store(below)`. The store below gives an object's document, by its path (`document(path)`): a one-transaction document of its properties and its entries, each object they link by its path. The store above loads an object the first time its values or entries are read, as its committed data, observed by none; an object an entry links that is not yet loaded is held by its path, the same one each time it is named, and loaded when it is read. Given no store below, a store holds what is written to it, as it always has.

In [ ]:
import json

from mbse.Schemas.Framework import JSON, Builders as B, Changes, Plain, References
from support import raises


def crm(txn):
    """crm.Person, owned by the singleton book by key, with a name and a home; persons know persons."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Home": S.OfObject, "Holds": S.OfRelation, "Knows": S.OfRelation, "Book": S.OfObject,
                  "Person": S.OfObject, "Keeps": S.OfRelation, "Pet": S.OfObject, "Walks": S.OfRelation})
        .spec([
            lambda _: _.Home.name("crm.Home").properties({"city": S.OfStr, "zip": S.OfStr}),
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")])
            .properties({"key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")]),
            lambda _: _.Keeps.name("crm.Keeps").links([lambda t: t.name("keeper"), lambda t: t.name("pet")])
            .properties({"name": S.OfStr}).uniques([["keeper", "name"], ["pet"]]),
            lambda _: _.Walks.name("crm.Walks").links([lambda t: t.name("walker"), lambda t: t.name("walked")]),
            lambda _: _.Pet.name("crm.Pet").ref(True).properties({"kind": S.OfStr}).adjacencies([
                lambda t: t.name("keeper").relation([_.Keeps]).me("pet").owner(True),
                lambda t: t.name("walkers").relation([_.Walks]).me("walked")]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr, "home": _.Home}).adjacencies([
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True),
                lambda t: t.name("knows").relation([_.Knows]).me("knower"),
                lambda t: t.name("known_by").relation([_.Knows]).me("known"),
                lambda t: t.name("pets").relation([_.Keeps]).me("keeper"),
                lambda t: t.name("walks").relation([_.Walks]).me("walker"),
            ]),
        ])]))])


below = B.Store()
below.transaction(crm)
book = below.singleton("crm.TheBook")
with below.transaction():
    people = {n: below.crm.Person(lambda p, n=n: p.book(lambda a: a.insert(lambda e: e.key(n).book(book))).name(n))
              for n in ("ada", "bob", "cy")}
    below.crm.Person(people["ada"], lambda p: p.home(lambda h: h.city("Paris").zip("75001"))
                     .knows(lambda a: a.insert(lambda e: e.known(people["bob"]))))
    rex_below = below.crm.Pet(lambda p: p.keeper(lambda a: a.insert(lambda e: e.name("rex").keeper(people["ada"])))
                              .kind("dog"))
    below.crm.Person(people["bob"], lambda p: p.walks(lambda a: a.insert(lambda e: e.walked(rex_below))))
above = B.Store(below)
above.transaction(crm)


def loaded():
    """How many objects of the store below's the store above has loaded."""
    return len(above._loaded)

<a id="BLW-01"></a>
## BLW-01 · Nothing is loaded until it is read; reading an object loads it alone, the objects its entries link each held by its path until it is read

In [ ]:
ours = above.singleton("crm.TheBook")
before = loaded()
keys = [e.key.get() for e in ours.contacts]
after_book = loaded()
ada = next(e.contact for e in ours.contacts if e.key.get() == "ada")
name = ada.name.get()
assert json.dumps([before, keys, after_book, name, loaded()]) == '[0, ["ada", "bob", "cy"], 1, "ada", 2]'

<a id="BLW-02"></a>
## BLW-02 · An object is the same object each time it is named, through any entry; its value objects and entries are loaded with it; an object reached by its reference loads with the owners its path passes through, and no other; one reached through an entry loads alone, its owner not loaded

In [ ]:
bob = next(e.contact for e in ours.contacts if e.key.get() == "bob")
assert next(e.known for e in ada.knows) is bob and next(e.book for e in ada.book) is ours
assert next(e.knower for e in bob.known_by) is ada
assert json.dumps([ada.home.city.get(), ada.home.zip.get(), bob.name.get(), ada.as_schema().name]) == '["Paris", "75001", "bob", "crm.Person"]'
fresh = B.Store(below)
fresh.transaction(crm)
rex_path = {"singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book", "properties": {"key": "ada"},
            "next": {"entry": {"i": "pets", "o": "keeper", "properties": {"name": "rex"}, "next": {"target": {}}}}}}}
rex = References.resolve(fresh, Plain.FromPlain(fresh)(fresh.registered("References.OfObject"), {"path": rex_path}))
assert json.dumps([len(fresh._loaded), repr(rex), rex.kind.get(), len(fresh._loaded)]) == '[3, "<crm.Pet>", "dog", 3]'
other = B.Store(below)
other.transaction(crm)
walker = next(e.contact for e in other.singleton("crm.TheBook").contacts if e.key.get() == "bob")
walked = next(e.walked for e in walker.walks)
assert json.dumps([len(other._loaded), repr(walked), walked.kind.get(), len(other._loaded)]) == '[2, "<crm.Pet>", "dog", 3]'

<a id="BLW-03"></a>
## BLW-03 · The store below gives an object's document by its path, each object it links by its path; none where it holds none, and reading such an object above is refused

In [ ]:
path = {"singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book", "properties": {"key": "cy"},
                                                       "next": {"target": {}}}}}
document = below.document(path)
nobody = {"singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book",
                                                          "properties": {"key": "zed"}, "next": {"target": {}}}}}
assert json.dumps([JSON.dumps(document), below.document(nobody)]) == '["{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"0\\": {\\"schema\\": [{\\"$ref\\": {\\"singleton\\": \\".\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"schemas\\", \\"o\\": \\"\\", \\"properties\\": {\\"name\\": \\"crm\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"schemas\\", \\"o\\": \\"\\", \\"properties\\": {\\"name\\": \\"Person\\"}, \\"next\\": {\\"target\\": {}}}}}}}}]}}, \\"spec\\": [{\\"0\\": [{\\"as_property\\": {\\"name\\": \\"name\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"cy\\"}}]}}, {\\"as_adjacency\\": {\\"name\\": \\"book\\", \\"mutations\\": [{\\"insert\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"book\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"target\\": {}}}}}}]}}, {\\"as_property\\": {\\"name\\": \\"key\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"cy\\"}}]}}]}]}}]}}]}]}]}}]}", null]'
with raises(LookupError, match="the store below holds no object at this path"):
    above._identity(nobody).name.get()

<a id="BLW-04"></a>
## BLW-04 · Loading is no transaction of the store's: its followers are given nothing of it, and a transaction open while an object loads keeps its own changes alone

In [ ]:
effects = []
end = Changes.observe(above, effects.append)
cy = next(e.contact for e in ours.contacts if e.key.get() == "cy")
with above.transaction():
    seen = cy.name.get()  # loaded within the transaction
    above.crm.Person(cy, lambda p: p.name("Cy"))
end()
(effect,) = effects
assert json.dumps([seen, cy.name.get(), JSON.dumps(Plain.ToPlain(above).Transaction(effect))]) == '["cy", "Cy", "{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"o0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"cy\\"}, \\"next\\": {\\"target\\": {}}}}}}}}, \\"spec\\": [{\\"o0\\": [{\\"as_property\\": {\\"name\\": \\"name\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"Cy\\"}}]}}]}]}]}}]}"]'

<a id="BLW-05"></a>
## BLW-05 · An object loaded has its reference as the store below has it; given no store below, a store holds what is written to it

In [ ]:
alone = B.Store()
alone.transaction(crm)
with alone.transaction():
    alone.crm.Person(lambda p: p.book(lambda a: a.insert(lambda e: e.key("dee").book(alone.singleton("crm.TheBook")))))
def path_of(store, x):
    """An object's reference, as plain data."""
    return JSON.dumps(Plain.ToPlain(store)(store.registered("References.OfObject"), References.of(store, x)))


assert json.dumps([path_of(above, cy) == path_of(below, people["cy"]),
                   [e.key.get() for e in alone.singleton("crm.TheBook").contacts], alone._below]) == '[true, ["dee"], null]'

<a id="BLW-06"></a>
## BLW-06 · An object the store above makes is found again by its path, and one the store below adds after its owner is loaded above is loaded when it is named

In [ ]:
with above.transaction():
    eve = above.crm.Person(lambda p: p.book(lambda a: a.insert(lambda e: e.key("eve").book(ours))).name("eve"))
found = References.resolve(above, References.of(above, eve))
with below.transaction():
    below.crm.Person(lambda p: p.book(lambda a: a.insert(lambda e: e.key("fay").book(book))).name("fay"))
fay_path = {"singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book", "properties": {"key": "fay"},
                                                           "next": {"target": {}}}}}
fay = References.resolve(above, Plain.FromPlain(above)(above.registered("References.OfObject"), {"path": fay_path}))
assert json.dumps([found is eve, fay.name.get()]) == '[true, "fay"]'